# 02. CPU 기준선 학습 및 평가

고정 fold에서 mean/surface/TF-IDF OOF와 validation 예측을 만들고, nested TF-IDF,
bootstrap 비교 및 OOF calibration 후보를 검증한다.


## 1. 독립 Colab 부트스트랩


In [ ]:
from __future__ import annotations

import importlib.util
import json
import os
import re
import shlex
import shutil
import subprocess
import sys
from pathlib import Path

IN_COLAB = importlib.util.find_spec("google.colab") is not None
REPO_URL = "https://github.com/Chika1472/Writing-validation.git"
# 이 노트북 변환 시점의 소스 commit. 새 코드를 사용할 때만 의도적으로 바꾼다.
REPO_REF = os.environ.get(
    "WRITING_VALIDATION_REPO_REF",
    "30c70cb628208e004515144cc999e81d794bbe95",
).strip()
COLAB_PROJECT_DIR = Path("/content/Writing-validation")
CLONE_IF_MISSING = True

# 긴 학습은 두 값을 True로 두어 세션이 끊겨도 동일 절대경로를 유지한다.
USE_GOOGLE_DRIVE = False
DRIVE_ARTIFACT_DIR = Path("/content/drive/MyDrive/Writing-validation/artifacts")
DRIVE_HF_HOME = Path("/content/drive/MyDrive/Writing-validation/huggingface")


def run_process(command, *, cwd=None, env=None):
    command = [str(part) for part in command]
    print("$", shlex.join(command))
    return subprocess.run(
        command,
        cwd=str(cwd) if cwd else None,
        env=env,
        check=True,
    )


if IN_COLAB:
    if USE_GOOGLE_DRIVE:
        from google.colab import drive

        drive.mount("/content/drive")
    if not (COLAB_PROJECT_DIR / ".git").exists():
        if not CLONE_IF_MISSING:
            raise FileNotFoundError(f"저장소가 없습니다: {COLAB_PROJECT_DIR}")
        run_process(["git", "clone", REPO_URL, COLAB_PROJECT_DIR])
        run_process(["git", "checkout", "--detach", REPO_REF], cwd=COLAB_PROJECT_DIR)
    PROJECT_ROOT = COLAB_PROJECT_DIR.resolve()
else:
    candidates = [Path.cwd(), Path.cwd().parent]
    PROJECT_ROOT = next(
        (candidate.resolve() for candidate in candidates if (candidate / "pyproject.toml").is_file()),
        None,
    )
    if PROJECT_ROOT is None:
        raise FileNotFoundError("pyproject.toml이 있는 저장소 루트에서 실행하세요.")

os.chdir(PROJECT_ROOT)

if USE_GOOGLE_DRIVE:
    DRIVE_ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
    artifact_link = PROJECT_ROOT / "artifacts"
    if artifact_link.exists() and not artifact_link.is_symlink():
        raise RuntimeError(
            "artifacts가 이미 실제 디렉터리입니다. 산출물을 이동/삭제하지 말고 "
            "새 RUN_NAMESPACE를 사용해 경로 계약을 유지하세요."
        )
    if not artifact_link.exists():
        artifact_link.symlink_to(DRIVE_ARTIFACT_DIR, target_is_directory=True)
    DRIVE_HF_HOME.mkdir(parents=True, exist_ok=True)
    os.environ["HF_HOME"] = str(DRIVE_HF_HOME)

# 변환 전 commit의 옛 설정 경로도 Colab Linux에서 안전하게 호환한다.
compatibility_links = [
    (PROJECT_ROOT / "데이터셋", PROJECT_ROOT / "dataset"),
    (PROJECT_ROOT / "qwen3_14b_zero_shot", PROJECT_ROOT / "result" / "qwen3_14b_zero_shot"),
]
if IN_COLAB:
    for alias, target in compatibility_links:
        if not alias.exists() and target.exists():
            alias.symlink_to(target, target_is_directory=True)


def run_cli(label: str, enabled: bool, *arguments: object, env=None):
    command = [sys.executable, *[str(argument) for argument in arguments]]
    print(f"\n[{label}]", "RUN" if enabled else "SKIP")
    print("$", shlex.join(command))
    if not enabled:
        return None
    return subprocess.run(command, cwd=PROJECT_ROOT, env=env, check=True)


def require_paths(*paths: Path) -> None:
    missing = [str(path) for path in paths if not Path(path).exists()]
    if missing:
        raise FileNotFoundError("필수 파일이 없습니다: " + ", ".join(missing))


def require_model_revision(value: str, *, enabled: bool) -> None:
    if enabled and re.fullmatch(r"[0-9a-fA-F]{40}", value or "") is None:
        raise ValueError("MODEL_REVISION에 40자리 Hugging Face commit SHA를 입력하세요.")


def require_bf16_gpu(*, enabled: bool, recommended_vram_gb: float = 22.0) -> None:
    if not enabled:
        print("GPU 단계가 꺼져 있어 BF16 검사를 건너뜁니다.")
        return
    import torch

    if not torch.cuda.is_available():
        raise RuntimeError("CUDA GPU가 필요합니다. Colab 런타임을 GPU로 변경하세요.")
    if not torch.cuda.is_bf16_supported():
        raise RuntimeError("이 파이프라인은 BF16이 필요합니다. T4 대신 L4/A100급을 선택하세요.")
    properties = torch.cuda.get_device_properties(0)
    vram_gb = properties.total_memory / 1024**3
    print({"gpu": properties.name, "vram_gb": round(vram_gb, 1), "bf16": True})
    if vram_gb < recommended_vram_gb:
        print(f"경고: 권장 VRAM {recommended_vram_gb:.0f}GB보다 작아 OOM 가능성이 큽니다.")


def show_json(path: Path):
    path = Path(path)
    if not path.exists():
        print("없음:", path)
        return None
    payload = json.loads(path.read_text(encoding="utf-8"))
    print(json.dumps(payload, ensure_ascii=False, indent=2)[:12000])
    return payload


commit = subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=PROJECT_ROOT, text=True
).strip()
if IN_COLAB and commit != REPO_REF:
    raise RuntimeError(
        f"기존 Colab checkout {commit}이 고정 REPO_REF {REPO_REF}와 다릅니다. "
        "새 런타임/새 PROJECT_DIR를 사용하거나 모든 노트북에서 같은 ref를 지정하세요."
    )
print({
    "in_colab": IN_COLAB,
    "project_root": str(PROJECT_ROOT),
    "git_commit": commit,
    "artifacts": str((PROJECT_ROOT / "artifacts").resolve()),
    "hf_home": os.environ.get("HF_HOME"),
})


In [ ]:
INSTALL_DEPENDENCIES = IN_COLAB
PROJECT_EXTRAS = "test"

if INSTALL_DEPENDENCIES:
    run_process(
        [sys.executable, "-m", "pip", "install", "-q", "-e", f".[{PROJECT_EXTRAS}]"],
        cwd=PROJECT_ROOT,
    )
else:
    print("로컬 검증에서는 기존 환경을 사용합니다.")


## 2. 입력, 출력 namespace 및 실행 스위치


In [ ]:
RUN_NAMESPACE = "cpu_v1"
FOLDS_PATH = PROJECT_ROOT / "artifacts" / "folds" / "folds_5fold_seed42.jsonl"
TRAIN_PATH = PROJECT_ROOT / "dataset" / "글쓰기채점능력평가2026_train.jsonl"
VALIDATION_PATH = PROJECT_ROOT / "dataset" / "글쓰기채점능력평가2026_validation.jsonl"
BASELINE_DIR = PROJECT_ROOT / "artifacts" / "predictions" / RUN_NAMESPACE
NESTED_DIR = PROJECT_ROOT / "artifacts" / "predictions" / f"{RUN_NAMESPACE}_nested"
TEST_PATH = PROJECT_ROOT / "dataset" / "test.jsonl"  # 외부에서 제공해야 함

RUN_BASELINES = False
RUN_NESTED_TFIDF = False
RUN_VALIDATION_EVALUATION = False
RUN_BOOTSTRAP_COMPARISON = False
RUN_TFIDF_CALIBRATION = False
RUN_TEST_PREDICTION = False

require_paths(FOLDS_PATH, TRAIN_PATH, VALIDATION_PATH)
print({"baseline_dir": str(BASELINE_DIR), "nested_dir": str(NESTED_DIR)})


## 3. 고정 CPU 기준선 및 nested TF-IDF


In [ ]:
run_cli(
    "fixed CPU baselines",
    RUN_BASELINES,
    "scripts/run_baselines.py",
    "--config", "configs/data.yaml",
    "--baseline-config", "configs/baselines.yaml",
    "--folds", FOLDS_PATH,
    "--output-dir", BASELINE_DIR,
)
run_cli(
    "nested TF-IDF",
    RUN_NESTED_TFIDF,
    "scripts/run_nested_tfidf.py",
    "--config", "configs/tfidf_nested.yaml",
    "--data-config", "configs/data.yaml",
    "--folds", FOLDS_PATH,
    "--output-dir", NESTED_DIR,
    "--model-name", f"nested_tfidf_{RUN_NAMESPACE}",
)
show_json(BASELINE_DIR / "metrics.json")


## 4. validation 평가와 paired bootstrap


In [ ]:
TFIDF_VALIDATION = BASELINE_DIR / "tfidf_ridge_validation.jsonl"
FIXED_TFIDF_OOF = BASELINE_DIR / "tfidf_ridge_oof.jsonl"
NESTED_OOF = NESTED_DIR / "nested_tfidf_oof.jsonl"

run_cli(
    "validation metrics",
    RUN_VALIDATION_EVALUATION,
    "scripts/evaluate_predictions.py",
    "--config", "configs/data.yaml",
    "--gold", VALIDATION_PATH,
    "--pred", TFIDF_VALIDATION,
    "--output", PROJECT_ROOT / "artifacts" / "reports" / f"{RUN_NAMESPACE}_validation.json",
    "--strict",
)
run_cli(
    "nested vs fixed bootstrap",
    RUN_BOOTSTRAP_COMPARISON,
    "scripts/bootstrap_compare.py",
    "--gold", TRAIN_PATH,
    "--candidate", NESTED_OOF,
    "--baseline", FIXED_TFIDF_OOF,
    "--output", PROJECT_ROOT / "artifacts" / "reports" / f"{RUN_NAMESPACE}_nested_vs_fixed.json",
)


## 5. OOF calibration 및 외부 test 적용


In [ ]:
CALIBRATOR = PROJECT_ROOT / "artifacts" / "calibrators" / f"{RUN_NAMESPACE}_tfidf_affine.json"
CALIBRATED_OOF = PROJECT_ROOT / "artifacts" / "predictions" / f"{RUN_NAMESPACE}_tfidf_crossfit_calibrated.jsonl"

run_cli(
    "TF-IDF OOF calibration",
    RUN_TFIDF_CALIBRATION,
    "scripts/fit_calibrator.py",
    "--config", "configs/data.yaml",
    "--gold", TRAIN_PATH,
    "--pred", FIXED_TFIDF_OOF,
    "--folds", FOLDS_PATH,
    "--output", CALIBRATOR,
    "--calibrated-output", CALIBRATED_OOF,
)

if RUN_TEST_PREDICTION:
    require_paths(TEST_PATH, BASELINE_DIR / "tfidf_ridge_ensemble.json")
run_cli(
    "TF-IDF test inference",
    RUN_TEST_PREDICTION,
    "scripts/predict_baseline.py",
    "--input", TEST_PATH,
    "--model", BASELINE_DIR / "tfidf_ridge_ensemble.json",
    "--output", PROJECT_ROOT / "artifacts" / "predictions" / f"{RUN_NAMESPACE}_tfidf_test.jsonl",
)


## 6. 승격 판단

validation을 보며 hyperparameter를 고르지 않는다. nested 후보는 train OOF paired
bootstrap에서 고정 TF-IDF보다 낫고 provenance 검사를 통과할 때만 이후 stacker의
`tfidf` source로 사용한다.
